In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)


# ============================================================
# 1) PARAMETERS
# ============================================================

def get_params():
    media_channels = ["tv", "search", "social", "video"]
    non_media_channels = ["promo_index", "distribution_index", "seasonality_index"]

    # adstock decay
    adstock_alpha = {
        "tv": 0.75, "search": 0.45, "social": 0.35, "video": 0.55,
        "promo_index": 0.20, "distribution_index": 0.60, "seasonality_index": 0.10
    }

    # hill params (units space)
    hill_beta = {
        "tv": 1200, "search": 900, "social": 700, "video": 800,
        "promo_index": 500, "distribution_index": 450, "seasonality_index": 350
    }
    hill_k = {
        "tv": 1.6, "search": 1.2, "social": 1.0, "video": 1.3,
        "promo_index": 0.8, "distribution_index": 1.0, "seasonality_index": 0.7
    }
    hill_s = {
        "tv": 1.7, "search": 1.4, "social": 1.6, "video": 1.5,
        "promo_index": 1.2, "distribution_index": 1.1, "seasonality_index": 1.0
    }

    # priors
    # media ROI priors (revenue/spend)
    roi_priors_media = {"tv": 1.8, "search": 3.0, "social": 2.4, "video": 2.0}

    # non-media contribution priors (share)
    contribution_priors_non_media = {
        "promo_index": 0.45,
        "distribution_index": 0.35,
        "seasonality_index": 0.20
    }

    return {
        "n_periods": 156,
        "freq": "W",
        "media_channels": media_channels,
        "non_media_channels": non_media_channels,
        "adstock_alpha": adstock_alpha,
        "hill_beta": hill_beta,
        "hill_k": hill_k,
        "hill_s": hill_s,
        "roi_priors_media": roi_priors_media,
        "contribution_priors_non_media": contribution_priors_non_media
    }


# ============================================================
# 2) TRANSFORMS
# ============================================================

def geometric_adstock(x, alpha):
    out = np.zeros_like(x, dtype=float)
    out[0] = x[0]
    for t in range(1, len(x)):
        out[t] = x[t] + alpha * out[t - 1]
    return out


def hill_transform(x, beta, k, s):
    x_safe = np.clip(x, 1e-9, None)
    num = np.power(x_safe, s)
    den = num + np.power(k, s)
    return beta * (num / den)


def median_scale(x):
    med = np.median(x)
    if med == 0:
        return x.copy(), med
    return x / med, med


def median_inverse(x_scaled, med):
    return x_scaled * med


def zscore_scale(x):
    mu = np.mean(x)
    sd = np.std(x)
    if sd == 0:
        sd = 1.0
    return (x - mu) / sd, mu, sd


def zscore_inverse(z, mu, sd):
    return z * sd + mu


# ============================================================
# 3) GENERATE ORIGINAL DATA
# ============================================================

def generate_original_data(n_periods=156, freq="W"):
    idx = pd.date_range("2023-01-01", periods=n_periods, freq=freq)
    n = len(idx)
    t = np.arange(n)

    # media impressions
    tv_imp = 15_000_000 + 2_000_000 * np.sin(2 * np.pi * t / 52 + 0.3) + np.random.normal(0, 1_100_000, n)
    search_imp = 9_000_000 + 1_200_000 * np.sin(2 * np.pi * t / 26 + 0.8) + np.random.normal(0, 700_000, n)
    social_imp = 7_000_000 + 1_100_000 * np.sin(2 * np.pi * t / 13 + 1.2) + np.random.normal(0, 650_000, n)
    video_imp = 8_000_000 + 1_300_000 * np.sin(2 * np.pi * t / 39 + 0.5) + np.random.normal(0, 750_000, n)

    tv_imp = np.clip(tv_imp, 1_000_000, None)
    search_imp = np.clip(search_imp, 600_000, None)
    social_imp = np.clip(social_imp, 500_000, None)
    video_imp = np.clip(video_imp, 700_000, None)

    # CPM-based spend from impressions
    tv_cpm = np.random.normal(11.5, 0.7, n)
    search_cpm = np.random.normal(9.0, 0.5, n)
    social_cpm = np.random.normal(7.5, 0.4, n)
    video_cpm = np.random.normal(10.0, 0.6, n)

    tv_spend = (tv_imp / 1000.0) * tv_cpm
    search_spend = (search_imp / 1000.0) * search_cpm
    social_spend = (social_imp / 1000.0) * social_cpm
    video_spend = (video_imp / 1000.0) * video_cpm

    # non-media
    promo_index = np.clip(np.random.normal(0.5, 0.15, n), 0.05, 1.0)
    distribution_index = np.clip(0.75 + 0.05 * np.sin(2 * np.pi * t / 52 + 2.2) + np.random.normal(0, 0.03, n), 0.55, 0.95)
    seasonality_index = 0.5 + 0.5 * np.sin(2 * np.pi * t / 52 - 1.1)

    # price
    price = 9.5 + 0.35 * np.sin(2 * np.pi * t / 52 + 0.2) + np.random.normal(0, 0.1, n)
    price = np.clip(price, 8.8, 10.5)

    df = pd.DataFrame({
        "date": idx,

        "tv_impressions": tv_imp,
        "tv_spend": tv_spend,

        "search_impressions": search_imp,
        "search_spend": search_spend,

        "social_impressions": social_imp,
        "social_spend": social_spend,

        "video_impressions": video_imp,
        "video_spend": video_spend,

        "promo_index": promo_index,
        "distribution_index": distribution_index,
        "seasonality_index": seasonality_index,

        "price": price
    })

    return df


# ============================================================
# 4) FEATURE ENGINEERING (SCALING + ADSTOCK + HILL + INVERSE)
# ============================================================

def build_transforms(df, media_channels, non_media_channels, adstock_alpha, hill_beta, hill_k, hill_s):
    out = df.copy()

    media_medians = {}
    non_media_stats = {}

    # ----- MEDIA: use impressions -----
    for c in media_channels:
        col = f"{c}_impressions"

        # median scaling
        scaled, med = median_scale(out[col].values)
        media_medians[c] = med
        out[f"{c}_scaled"] = scaled

        # adstock + hill
        out[f"{c}_adstock"] = geometric_adstock(out[f"{c}_scaled"].values, adstock_alpha[c])
        out[f"{c}_hill_units"] = hill_transform(out[f"{c}_adstock"].values, hill_beta[c], hill_k[c], hill_s[c])

        # inverse scaling (store back-transformed signal)
        out[f"{c}_scaled_inverse"] = median_inverse(out[f"{c}_scaled"].values, med)

    # ----- NON-MEDIA: z-score scaling -----
    for c in non_media_channels:
        x = out[c].values
        z, mu, sd = zscore_scale(x)
        non_media_stats[c] = (mu, sd)
        out[f"{c}_scaled"] = z

        # shift positive before hill
        z_pos = z - np.min(z) + 1e-3
        out[f"{c}_adstock"] = geometric_adstock(z_pos, adstock_alpha[c])
        out[f"{c}_hill_units"] = hill_transform(out[f"{c}_adstock"].values, hill_beta[c], hill_k[c], hill_s[c])

        # inverse scaling
        out[f"{c}_scaled_inverse"] = zscore_inverse(out[f"{c}_scaled"].values, mu, sd)

    return out, media_medians, non_media_stats


# ============================================================
# 5) PRIORS -> COEFFICIENTS
# ============================================================

def media_coef_from_roi_priors(df_t, media_channels, roi_priors_media):
    # target incremental revenue for channel = ROI_prior * total_spend
    # channel contribution revenue = coef * sum(hill_units * price)
    coefs = {}
    for c in media_channels:
        target_revenue = roi_priors_media[c] * df_t[f"{c}_spend"].sum()
        denom = (df_t[f"{c}_hill_units"].values * df_t["price"].values).sum()
        coefs[c] = target_revenue / denom if denom > 0 else 0.0
    return coefs


def nonmedia_coef_from_contribution_priors(df_t, non_media_channels, contribution_priors_non_media, total_nonmedia_revenue_budget):
    priors = contribution_priors_non_media.copy()
    s = sum(priors.values())
    priors_norm = {k: v / s for k, v in priors.items()}

    coefs = {}
    for c in non_media_channels:
        target_revenue = priors_norm[c] * total_nonmedia_revenue_budget
        denom = (df_t[f"{c}_hill_units"].values * df_t["price"].values).sum()
        coefs[c] = target_revenue / denom if denom > 0 else 0.0
    return coefs


# ============================================================
# 6) BUILD CONTRIBUTIONS / REVENUE / ROI
# ============================================================

def build_outputs(
    df_t,
    media_channels,
    non_media_channels,
    roi_priors_media,
    contribution_priors_non_media,
    baseline_units=120_000,
    nonmedia_vs_media_ratio=0.35,
    noise_sd=0.03
):
    out = df_t.copy()

    coef_media = media_coef_from_roi_priors(out, media_channels, roi_priors_media)

    # media weekly contributions
    for c in media_channels:
        out[f"contrib_{c}_units"] = coef_media[c] * out[f"{c}_hill_units"]
        out[f"contrib_{c}_revenue"] = out[f"contrib_{c}_units"] * out["price"]

    # allocate non-media budget relative to media total contribution
    media_total_revenue = sum(out[f"contrib_{c}_revenue"].sum() for c in media_channels)
    total_nonmedia_revenue_budget = nonmedia_vs_media_ratio * media_total_revenue

    coef_nonmedia = nonmedia_coef_from_contribution_priors(
        out, non_media_channels, contribution_priors_non_media, total_nonmedia_revenue_budget
    )

    for c in non_media_channels:
        out[f"contrib_{c}_units"] = coef_nonmedia[c] * out[f"{c}_hill_units"]
        out[f"contrib_{c}_revenue"] = out[f"contrib_{c}_units"] * out["price"]

    all_channels = media_channels + non_media_channels
    contrib_rev_cols = [f"contrib_{c}_revenue" for c in all_channels]

    # total incremental revenue
    out["incremental_revenue"] = out[contrib_rev_cols].sum(axis=1)

    # baseline + observed
    out["baseline_revenue"] = baseline_units * out["price"]
    out["revenue_modeled_no_noise"] = out["baseline_revenue"] + out["incremental_revenue"]
    eps = np.random.normal(0, noise_sd, len(out))
    out["revenue_observed"] = out["revenue_modeled_no_noise"] * (1 + eps)

    # inverse through price (requested)
    out["units_observed"] = out["revenue_observed"] / out["price"]
    out["revenue_reconstructed"] = out["units_observed"] * out["price"]

    # ROI table (channel level)
    roi_rows = []
    for c in media_channels:
        total_contrib_rev = out[f"contrib_{c}_revenue"].sum()
        total_spend = out[f"{c}_spend"].sum()
        roi = total_contrib_rev / total_spend if total_spend > 0 else np.nan
        roi_rows.append({
            "channel": c,
            "type": "media",
            "total_spend": total_spend,
            "total_incremental_revenue": total_contrib_rev,
            "roi": roi
        })

    for c in non_media_channels:
        total_contrib_rev = out[f"contrib_{c}_revenue"].sum()
        roi_rows.append({
            "channel": c,
            "type": "non_media",
            "total_spend": np.nan,
            "total_incremental_revenue": total_contrib_rev,
            "roi": np.nan
        })

    roi_df = pd.DataFrame(roi_rows).sort_values(["type", "channel"]).reset_index(drop=True)

    # weekly contribution output
    weekly_cols = ["date", "price", "baseline_revenue", "incremental_revenue", "revenue_observed"]
    for c in all_channels:
        weekly_cols += [f"contrib_{c}_units", f"contrib_{c}_revenue"]
    weekly_df = out[weekly_cols].copy()

    return out, weekly_df, roi_df, coef_media, coef_nonmedia


# ============================================================
# 7) RESPONSE CURVES (ONE TAB PER MEDIA CHANNEL)
# ============================================================

def response_curve_media_channel(
    df_t,
    channel,
    medians,
    adstock_alpha,
    hill_beta,
    hill_k,
    hill_s,
    coef_media,
    n_points=60
):
    imp_col = f"{channel}_impressions"
    spend_col = f"{channel}_spend"

    p5_imp, p95_imp = np.percentile(df_t[imp_col].values, [5, 95])
    imp_grid = np.linspace(max(1.0, 0.5 * p5_imp), 1.5 * p95_imp, n_points)

    # impressions -> spend relationship via avg CPM
    avg_cpm = np.mean((df_t[spend_col].values / np.clip(df_t[imp_col].values, 1e-9, None)) * 1000.0)
    spend_grid = (imp_grid / 1000.0) * avg_cpm

    # scale + adstock (steady-state approx) + hill
    med = medians[channel]
    x_scaled = imp_grid / med if med != 0 else imp_grid
    x_ads_ss = x_scaled / (1 - adstock_alpha[channel])

    hill_units = hill_transform(x_ads_ss, hill_beta[channel], hill_k[channel], hill_s[channel])
    contrib_units = coef_media[channel] * hill_units

    avg_price = df_t["price"].mean()
    contrib_revenue = contrib_units * avg_price
    roi = contrib_revenue / np.clip(spend_grid, 1e-9, None)

    curve_df = pd.DataFrame({
        "impressions": imp_grid,
        "spend": spend_grid,
        "predicted_incremental_units": contrib_units,
        "predicted_incremental_revenue": contrib_revenue,
        "implied_roi": roi
    })
    return curve_df


def build_response_curves_excel(
    df_t,
    media_channels,
    medians,
    adstock_alpha,
    hill_beta,
    hill_k,
    hill_s,
    coef_media,
    out_xlsx="response_curves.xlsx"
):
    with pd.ExcelWriter(out_xlsx, engine="xlsxwriter") as writer:
        for c in media_channels:
            cdf = response_curve_media_channel(
                df_t=df_t,
                channel=c,
                medians=medians,
                adstock_alpha=adstock_alpha,
                hill_beta=hill_beta,
                hill_k=hill_k,
                hill_s=hill_s,
                coef_media=coef_media,
                n_points=60
            )
            sheet_name = c[:31]  # excel limit
            cdf.to_excel(writer, index=False, sheet_name=sheet_name)


# ============================================================
# 8) RUN ALL + SAVE FILES
# ============================================================

def run_pipeline():
    p = get_params()

    # original raw data
    original_data = generate_original_data(n_periods=p["n_periods"], freq=p["freq"])
    original_data.to_csv("./outputs/original_data.csv", index=False)

    # transformed
    transformed, media_medians, non_media_stats = build_transforms(
        df=original_data,
        media_channels=p["media_channels"],
        non_media_channels=p["non_media_channels"],
        adstock_alpha=p["adstock_alpha"],
        hill_beta=p["hill_beta"],
        hill_k=p["hill_k"],
        hill_s=p["hill_s"]
    )

    # outputs
    full_df, weekly_contribution, roi_df, coef_media, coef_nonmedia = build_outputs(
        df_t=transformed,
        media_channels=p["media_channels"],
        non_media_channels=p["non_media_channels"],
        roi_priors_media=p["roi_priors_media"],
        contribution_priors_non_media=p["contribution_priors_non_media"],
        baseline_units=120_000,
        nonmedia_vs_media_ratio=0.35,
        noise_sd=0.03
    )

    # save required CSVs
    weekly_contribution.to_csv("./outputs/weekly_contribution.csv", index=False)
    roi_df.to_csv("./outputs/roi.csv", index=False)

    # response curves one tab per media channel
    build_response_curves_excel(
        df_t=full_df,
        media_channels=p["media_channels"],
        medians=media_medians,
        adstock_alpha=p["adstock_alpha"],
        hill_beta=p["hill_beta"],
        hill_k=p["hill_k"],
        hill_s=p["hill_s"],
        coef_media=coef_media,
        out_xlsx="./outputs/response_curves.xlsx"
    )

    print("Saved files:")
    print("- original_data.csv")
    print("- weekly_contribution.csv")
    print("- roi.csv")
    print("- response_curves.xlsx")


if __name__ == "__main__":
    run_pipeline()

Saved files:
- original_data.csv
- weekly_contribution.csv
- roi.csv
- response_curves.xlsx
